In [ ]:
import numpy as np
from alkaid.codegen import RTLModel
from alkaid.converter import trace_model
from alkaid.trace import FVArray, trace
from qxgb.model import QXGBClassifier
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

In [2]:
data = fetch_openml('hls4ml_lhc_jets_hlf')
X, y = np.array(data['data']), data['target']
codecs = {'g': 0, 'q': 1, 'w': 2, 'z': 3, 't': 4}
y = np.array([codecs[i] for i in y])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3)
_min, _max = X_train.min(axis=0), X_train.max(axis=0)
X_train = np.floor((X_train - _min) / (_max - _min) * 255)
X_test = np.floor((X_test - _min) / (_max - _min) * 255)
np.savez('/tmp/jsc.npz', X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test)

In [ ]:
def run(n_estimators, max_depth, scale, bias, n_stages, clock_period):

    X_train, y_train, X_test, y_test = np.load('/tmp/jsc.npz').values()

    model = QXGBClassifier(
        scale=scale,
        bias=bias,
        num_class=5,
        n_estimators=n_estimators,
        max_depth=max_depth,
        eta=0.8,
    )
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    print(f'Test acc: {np.mean(model.predict(X_test) == y_test):.4f}')

    hw_bst = model.ibooster()
    inp = FVArray.new(16).quantize(0, 8, 0).as_new()
    _, out = trace_model(hw_bst, inputs=inp, mode='mux')
    comb = trace(inp, out)

    train_acc = np.mean(np.argmax(comb.predict(X_train), axis=1) == y_train)
    test_acc = np.mean(np.argmax(comb.predict(X_test, n_threads=1), axis=1) == y_test)
    print(f'HW Train acc: {train_acc:.4f}, Test acc: {test_acc:.4f}')
    # print(f'Estimated LUT: {comb.cost:.1f}')

    rtl = RTLModel(
        comb,
        f'/tmp/qbdt/jsc-{n_estimators=}-{max_depth=}-{scale=}-{bias=}',
        'model',
        n_stages=n_stages,
        clock_period=clock_period,
        clock_uncertainty=0,
        part_name='xcvu9p-flgb2104-2-i',
    )
    rtl.write(xls_opt=True, metadata={'comb_metric': test_acc})
    for _ in range(6):
        try:
            rtl._compile(_env={'VERILATOR_FLAGS': ''}, nproc=1)
            break
        except Exception as _e:
            pass  # verilator internal error
    else:
        raise RuntimeError('Failed to compile RTL model after 6 attempts')
    assert np.all(rtl.predict(X_test, n_threads=1) == comb.predict(X_test, n_threads=1))  # bit-exact check


In [4]:
run(24, 4, 3, -2.5, 2, 2)

Test acc: 0.7574
HW Train acc: 0.7568, Test acc: 0.7569


In [5]:
run(12, 3, 3, -2.5, 1, 2)

Test acc: 0.7479
HW Train acc: 0.7467, Test acc: 0.7480
